# Week 6 -- Function 3: GP + UCB Exploitation (3D)
**Drug Discovery** — minimise adverse reaction score (maximise Y toward zero)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 15
DATA_PATH_X  = '../data/function_3/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_3/initial_outputs.npy'

# Key pattern: lower x3 consistently gives better (less negative) Y
# W1 x3=1.00 -> -0.476 | W2 x3=0.68 -> -0.196 | W3 x3=0.68 -> -0.160
# W4 x3=0.19 -> -0.126 | W5 x3=0.22 -> -0.118  (best weekly query)
weekly_log = [
    ([0.421053, 1.000000, 1.000000], -0.47622948272340854),    # W1: x3=1.00 -- worst
    ([1.000000, 0.000000, 0.684211], -0.1955411421962853),     # W2: x3=0.68 -- improved
    ([0.000000, 1.000000, 0.684211], -0.1602564778875669),     # W3: x3=0.68 -- improved
    ([0.692581, 0.411593, 0.194985], -0.12553372160833784),    # W4: x3=0.19 -- improved
    ([0.524514, 0.731593, 0.220176], -0.11825142496697436),    # W5: x3=0.22 -- BEST weekly
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Already up-to-date.
X: (20, 3) | Y: (20,)
Week 6 | 20 total observations (15 initial + 5 added)


In [2]:
# Cell 3: Load data and inspect -- Function 3: Drug Discovery (3D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

# Sort by Y descending -- less negative = better (want closest to zero)
pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 80)
print('  All observations (sorted descending by Y)   x3 column shows key pattern')
print('=' * 80)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6f}   x3 = {x_val[2]:.3f}{marker}')
print('=' * 80)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.6f}  (least negative)')
print(f'  Best X*  = [{best_x_str}]')
print(f'  x3*      = {best_X[2]:.6f}')

Input  shape : (20, 3)
Output shape : (20,)

  All observations (sorted descending by Y)   x3 column shows key pattern
  [ 1]  X = [0.492581, 0.611593, 0.340176]   Y = -0.034835   x3 = 0.340  <-- best
  [ 2]  X = [0.600097, 0.725136, 0.066089]   Y = -0.036378   x3 = 0.066
  [ 3]  X = [0.220549, 0.297825, 0.343555]   Y = -0.046947   x3 = 0.344
  [ 4]  X = [0.134622, 0.219917, 0.458206]   Y = -0.048008   x3 = 0.458
  [ 5]  X = [0.965995, 0.861120, 0.566829]   Y = -0.056758   x3 = 0.567
  [ 6]  X = [0.242114, 0.644074, 0.272433]   Y = -0.087963   x3 = 0.272
  [ 7]  X = [0.170477, 0.697032, 0.149169]   Y = -0.094190   x3 = 0.149
  [ 8]  X = [0.666014, 0.671985, 0.246295]   Y = -0.105965   x3 = 0.246
  [ 9]  X = [0.345523, 0.941360, 0.269363]   Y = -0.110621   x3 = 0.269
  [10]  X = [0.534906, 0.398501, 0.173389]   Y = -0.111415   x3 = 0.173
  [11]  X = [0.171525, 0.343917, 0.248737]   Y = -0.112122   x3 = 0.249
  [12]  X = [0.645503, 0.397143, 0.919771]   Y = -0.113869   x3 = 0.920
  [13] 

In [3]:
# Cell 4: Fit GP
# F3 values are all in a normal range (-0.035 to -0.476).
# Fit on raw Y directly -- no log transform needed.
# length_scale=0.3: wider kernel so GP extrapolates into the focused search region.
# alpha=1e-4: slight noise tolerance (consistent with other functions).
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

kernel = RBF(length_scale=0.3, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-4)
gp.fit(X, Y)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Alpha (noise)           : 1e-4')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.6f}')
print(f'    Actual Y*              = {best_Y:.6f}')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.3)
  Alpha (noise)           : 1e-4
  Log-marginal-likelihood : -8.4520
  Sanity check at best known X*:
    X*                     = [0.492581, 0.611593, 0.340176]
    GP predicted mean      = -0.034970
    Actual Y*              = -0.034835
    GP predicted std       = 0.00999192


In [4]:
# Cell 5: GP UCB Acquisition -- focused random search around initial data best
# FIX: 30^3 meshgrid had corner artefact -- UCB picked [0.40, 0.80, 0.000] (all grid edges).
# Root cause: GP uncertainty at corners inflates UCB beyond interior predictions (same as F6).
# NEW: Random search ±0.10 around best_X has no systematic grid edges.
# Anchor: best_X=[0.493, 0.612, 0.340] (initial data, Y=-0.035) -- NOT W5 (Y=-0.118, 3x worse).

np.random.seed(42)
X_grid = np.clip(
    best_X + np.random.uniform(-0.10, 0.10, size=(50000, n_dims)),
    0.0, 1.0
)

gp_mean, gp_std = gp.predict(X_grid, return_std=True)

# beta=1.0: exploit the known best region
beta = 1.0
ucb_scores = gp_mean + beta * gp_std

best_idx = np.argmax(ucb_scores)
next_x = X_grid[best_idx]
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 62)
print('  GP UCB Acquisition (beta=1.0, focused random search)')
print('=' * 62)
best_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Anchor       : [{best_str}]  (initial data best, Y={best_Y:.6f})')
print(f'  Search radius: ±0.10  |  50,000 random points  (no grid corners)')
print(f'  Beta         : {beta}  (exploitation)')
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query   : [{next_str}]')
print(f'  x3 proposed  : {next_x[2]:.6f}')
print(f'  UCB score    : {ucb_scores[best_idx]:.6f}')
print(f'  GP mean      : {gp_mean[best_idx]:.6f}')
print(f'  GP std       : {gp_std[best_idx]:.6f}')
print('=' * 62)

  GP UCB Acquisition (beta=1.0, focused random search)
  Anchor       : [0.492581, 0.611593, 0.340176]  (initial data best, Y=-0.034835)
  Search radius: ±0.10  |  50,000 random points  (no grid corners)
  Beta         : 1.0  (exploitation)
  Next query   : [0.582160, 0.512349, 0.435542]
  x3 proposed  : 0.435542
  UCB score    : 0.284797
  GP mean      : 0.042020
  GP std       : 0.242777


In [5]:
# Cell 6: Sanity check -- distance, x3 range, no exact-boundary values

dist_to_best  = np.linalg.norm(next_x - best_X)
dim_diffs     = np.abs(next_x - best_X)
x3_in_range   = 0.15 <= next_x[2] <= 0.45   # low but not zero -- initial best had x3=0.340
extreme_dims  = [i + 1 for i, v in enumerate(next_x) if v <= 0.001 or v >= 0.999]
dim_violations = [i + 1 for i, d in enumerate(dim_diffs) if d > 0.15]

print('=' * 62)
print('  Sanity Check')
print('=' * 62)
next_str = ', '.join(f'{v:.6f}' for v in next_x)
best_str  = ', '.join(f'{v:.6f}' for v in best_X)
print(f'  Proposed query  : [{next_str}]')
print(f'  Anchor (best X*): [{best_str}]  (Y={best_Y:.6f})')
print(f'  Distance to X*  : {dist_to_best:.6f}')
print()
dim_names = ['x1', 'x2', 'x3']
for name, a_val, n_val, diff in zip(dim_names, best_X, next_x, dim_diffs):
    ok = diff <= 0.15
    flag = '' if ok else '  <-- WARNING: > 0.15 from anchor'
    print(f'  {name}: anchor={a_val:.6f}  query={n_val:.6f}  diff={diff:.4f}{flag}')
print()
print(f'  x3 in [0.15, 0.45] : {x3_in_range}  (x3={next_x[2]:.6f})')
print()

if dist_to_best > 0.15:
    print(f'  WARNING: distance {dist_to_best:.4f} > 0.15 -- drifting from initial best.')
else:
    print(f'  OK: distance {dist_to_best:.4f} within 0.15 of anchor.')

if extreme_dims:
    print(f'  WARNING: dim(s) {extreme_dims} at or near 0.0 or 1.0 -- corners perform poorly!')
else:
    print(f'  OK: no dimensions at boundary extremes (all between 0.001 and 0.999).')

if not x3_in_range:
    print(f'  WARNING: x3={next_x[2]:.6f} outside [0.15, 0.45] -- initial best had x3=0.340.')
else:
    print(f'  OK: x3 is in the reasonable range [0.15, 0.45].')

if dim_violations:
    print(f'  WARNING: dim(s) {dim_violations} deviate > 0.15 from anchor.')
else:
    print(f'  OK: all dimensions within 0.15 of anchor.')
print('=' * 62)

  Sanity Check
  Proposed query  : [0.582160, 0.512349, 0.435542]
  Anchor (best X*): [0.492581, 0.611593, 0.340176]  (Y=-0.034835)
  Distance to X*  : 0.164221

  x1: anchor=0.492581  query=0.582160  diff=0.0896
  x2: anchor=0.611593  query=0.512349  diff=0.0992
  x3: anchor=0.340176  query=0.435542  diff=0.0954

  x3 in [0.15, 0.45] : True  (x3=0.435542)

  OK: no dimensions at boundary extremes (all between 0.001 and 0.999).
  OK: x3 is in the reasonable range [0.15, 0.45].
  OK: all dimensions within 0.15 of anchor.


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 6 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 3 -- Drug Discovery (3D)')
print('  Objective       : Maximise Y (all negative, want least negative = fewest side effects)')
print(f'  Method          : GP UCB (beta=1.0, random search ±0.10 of best_X, raw Y, ls=0.3)')
print(f'  Key pattern     : lower x3 -> better Y  (W1 x3=1.00 worst, initial best x3=0.34)')
print(f'  Anchor          : initial data best [0.493, 0.612, 0.340] at Y=-0.035 (NOT weekly)')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6f}')
print(f'  Best x3*        : {best_X[2]:.6f}')
print()
print(f'  Next query      : [{next_s}]')
print(f'  Next x3         : {next_x[2]:.6f}')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 6 Bayesian Optimisation
  Function        : 3 -- Drug Discovery (3D)
  Objective       : Maximise Y (all negative, want least negative = fewest side effects)
  Method          : GP UCB (beta=1.0, random search ±0.10 of best_X, raw Y, ls=0.3)
  Key pattern     : lower x3 -> better Y  (W1 x3=1.00 worst, initial best x3=0.34)
  Anchor          : initial data best [0.493, 0.612, 0.340] at Y=-0.035 (NOT weekly)

  Best X*         : [0.492581, 0.611593, 0.340176]
  Best Y*         : -0.034835
  Best x3*        : 0.340176

  Next query      : [0.582160, 0.512349, 0.435542]
  Next x3         : 0.435542

  Portal submission string:
  >>> 0.582160-0.512349-0.435542 <<<


## Week 6 -- Reflection

**Function**: F3 -- Drug Discovery (3D)

### Strategy change from Week 5
- Removed NN surrogate and SVM.
- No log transform: raw Y fit, alpha=1e-6.
- Focused 3D grid x1:[0.30,0.80] x x2:[0.50,1.00] x x3:[0.00,0.30] — 27,000 pts.
- x3 pushed toward zero: pattern W1→W5 shows lower x3 consistently improves Y.
- Beta=1.0: exploit the identified trend, don't explore corners.

### x3 pattern (weekly queries)
| Week | x3 | Y |
|------|----|---|
| W1 | 1.000 | -0.476 |
| W2 | 0.684 | -0.196 |
| W3 | 0.684 | -0.160 |
| W4 | 0.195 | -0.126 |
| W5 | 0.220 | -0.118 |

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 7
*(fill in after reflecting on result)*